In [2]:
import pandas as pd
import duckdb
import os
import glob

In [3]:
#read files into dictionary
dfs = {}
for file in glob.glob(os.path.join("clean_data", "*.parquet")): #search for all parquet files in clean_data, and iterate
    name = os.path.basename(file).replace(".parquet", "")
    dfs[name] = pd.read_parquet(file)

ANALYSIS 1: HAVE SPECIALISED MOLDS REPLACED ASSEMBLIES OF SMALLER PARTS TO CREATE THE SAME SHAPE? 
...OR VICE VERSA?

THE 'PART_RELATIONSHIPS' TABLE SHOWS 'PARENT' PARTS THAT HAVE RELATED 'CHILD' PARTS.
THESE 'CHILD' PARTS ARE EITHER SEGMENTS THAT COMBINE TO MAKE UP THE PARENT, 
OR THEY ARE A COMINBATION OF MULTIPLE child PARTS.
EITHER WAY, THE child(S) ARE THE PART(S) THAT WERE RELEASED FIRST.

TABLES NEEDED (6 out of 12):

    part_relationships - core table to identify child-child connections
    parts - to get the part name, for human readability
    part_categories - allow for more granular analysis by part category
    inventory_parts, inventories, sets - the 'year' column in the 'sets' table is needed to get the timeline of parent and child part usage


In [4]:
part_relationships = dfs["part_relationships"]
parts = dfs["parts"]
part_categories = dfs["part_categories"]
inventory_parts = dfs["inventory_parts"]
inventories = dfs["inventories"]
inventory_sets = dfs["inventory_sets"]
sets = dfs["sets"]

In [5]:
#only look at the parts that have a parent-child relatiosnhip, not eg. a mold update or print
part_relationships = duckdb.sql("SELECT * FROM part_relationships WHERE rel_type = 'R'").df()

In [6]:
#joined table is symmetrical in structure, where parent part info starts from the middle and continues through the left columns,
#and child part indo starts from the middle and continues through the right columnd
parent_child_parts = duckdb.sql("""
                    
                    WITH parts_sets AS(
                        SELECT
                            p.part_num,
                            s.set_num,
                            s.year,
                            ivs.quantity
                        FROM parts p JOIN inventory_parts ip ON p.part_num = ip.part_num
                        JOIN inventories i ON ip.inventory_id = i.id
                        JOIN inventory_sets ivs ON i.set_num = ivs.set_num
                        JOIN sets s ON ivs.set_num = s.set_num
                    )

                    SELECT DISTINCT ON(pr.parent_part_num, pr.child_part_num)
                                
                        pr.parent_part_num || pr.child_part_num AS id,
                        
                        MIN(parent_ps.year) OVER(PARTITION BY pr.parent_part_num, pr.child_part_num) AS min_parent_year,
                        MAX(parent_ps.year) OVER(PARTITION BY pr.parent_part_num, pr.child_part_num) AS max_parent_year,
                        pr.parent_part_num,
                    
                        pr.child_part_num,
                        MIN(child_ps.year) OVER(PARTITION BY pr.parent_part_num, pr.child_part_num) AS min_child_year,
                        MAX(child_ps.year) OVER(PARTITION BY pr.parent_part_num, pr.child_part_num) AS max_child_year

                    FROM part_relationships pr

                    JOIN parts child_p ON pr.child_part_num = child_p.part_num
                    JOIN parts parent_p ON pr.parent_part_num = parent_p.part_num

                    JOIN parts_sets parent_ps ON parent_p.part_num = parent_ps.part_num
                    JOIN parts_sets child_ps ON child_p.part_num = child_ps.part_num

                    WHERE pr.rel_type = 'R' --only ~8 percent of all parts have this relationship
                                
                    ORDER BY pr.parent_part_num, parent_ps.year ASC, child_ps.year ASC
                    
                    """).df() #takes 3.2 - 3.3 seconds to executed

In [7]:
parent_child_parts


,id,min_parent_year,max_parent_year,parent_part_num,child_part_num,min_child_year,max_child_year
0,1112611767,2014,2022,11126,11767,2014,2014
1,1112671100,2014,2022,11126,71100,2022,2022
2,1120811209,2014,2024,11208,11209,2014,2023
3,112085033,2014,2024,11208,5033,2024,2024
4,1121361485,2013,2025,11213,61485,2008,2026
...,...,...,...,...,...,...,...
973,upn0027pr000171308,1992,1992,upn0027pr0001,71308,1992,1992
974,upn0027pr0001upn0026pr0001,1992,1992,upn0027pr0001,upn0026pr0001,1992,1992
975,upn003571094,1990,1995,upn0035,71094,1990,1995
976,upn0068upn0182,2013,2013,upn0068,upn0182,2013,2013


- DISCOVERY: ACROSS THE BOARD, THE DATABSSE DOES NOT CONSIDER TWO PARTS THAT STRUCTURALLY COMBINE TO MAKE THE THIRD AS A 'PARENT-CHILD' RELATIONSHIP 
    - IN FACT, THERE IS NO RELATIOSNHIP WHATSOEVER. EG. BETWEEN PART 51739, AND 24299 WITH 24307. THIS IS LIKELY BECAUSE THE THE CONNECTIONS UNDERNEATH THE PIECES DIFFER. 
- FIX: I WILL HAVE TO USE MY OWN KNOWLEDGE OF PARTS TO PIECE TOGETHER MY ANALYSIS

In [8]:
mydf = duckdb.sql("SELECT * FROM part_relationships WHERE parent_part_num = '51739' OR child_part_num = '51739'")
mydf

┌──────────┬────────────────┬─────────────────┐
│ rel_type │ child_part_num │ parent_part_num │
│ varchar  │    varchar     │     varchar     │
└──────────┴────────────────┴─────────────────┘
                    0 rows                   

- DISCOVERY: SOME SPECIFIC INSTANCES OF PARENT-CHILD PART RELATIONSHIPS ARE PRESENT AFTER SOME MANUAL INSPECTIONS
- FIX: EXTRACT THE MAJORITY BY EXCLUDING WHERE MINIMUM PARENT YEAR = MINIMUM CHILD YEAR...
    - ...BECAUSE PARTS THAT WERE CREATED IN THE SAME YEAR ARE LIKELY TO BE PART MIRRORINGS, RATHER THAN 'TRUE' PARENT AND CHILD PARTS

In [9]:
parent_child_parts = duckdb.sql("SELECT * FROM parent_child_parts WHERE min_parent_year != min_child_year").df()
parent_child_parts

,id,min_parent_year,max_parent_year,parent_part_num,child_part_num,min_child_year,max_child_year
0,1112671100,2014,2022,11126,71100,2022,2022
1,112085033,2014,2024,11208,5033,2024,2024
2,1121361485,2013,2025,11213,61485,2008,2026
3,1121327448,2013,2025,11213,27448,2019,2025
4,1129987913,2013,2016,11299,87913,2010,2016
...,...,...,...,...,...,...,...
480,98374pr000195344,2013,2013,98374pr0001,95344,2011,2024
481,98374pr000295344,2015,2015,98374pr0002,95344,2011,2024
482,98374pr000395344,2016,2016,98374pr0003,95344,2011,2024
483,9845961649,2010,2018,98459,61649,2007,2018


In [10]:
#for each row of parent and child parts, find the quantity of each part used across all sets, for each year that part was in production

part_qty_per_year = duckdb.sql(
""" 
    WITH generic_part_totals AS(
        SELECT 
            ip.part_num, -- 'parts' table not needed as part_num is in 'inventory_parts'!
            s.year,
            SUM(ip.quantity)::int AS quantity
        FROM inventory_parts ip
        JOIN inventories i ON ip.inventory_id = i.id
        JOIN sets s ON i.set_num = s.set_num
        GROUP BY ip.part_num, s.year
    ),

    valid_years_per_relationship AS(
        SELECT DISTINCT ON (pcp.parent_part_num, pcp.child_part_num, gpt.year)
            pcp.parent_part_num,
            pcp.child_part_num,
            gpt.year
        FROM parent_child_parts pcp JOIN generic_part_totals gpt
        ON pcp.parent_part_num = gpt.part_num OR pcp.child_part_num = gpt.part_num   --the DISTINCT will discard the duplicates
    )

    --combining 'generic_part_totals' and 'valid_years_per_relationship':

    SELECT

        vy.parent_part_num || vy.child_part_num AS id, 

        vy.year,

        vy.parent_part_num,
        vy.child_part_num,

        COALESCE(parent_pt.quantity, 0) AS parent_qty,
        COALESCE(child_pt.quantity, 0) AS child_qty -- coalesce handles gaps in overlapping part timelines 

    FROM valid_years_per_relationship vy

    LEFT JOIN generic_part_totals parent_pt ON vy.parent_part_num = parent_pt.part_num
    AND vy.year = parent_pt.year

    LEFT JOIN generic_part_totals child_pt ON vy.child_part_num = child_pt.part_num
    AND vy.year = child_pt.year

    ORDER BY parent_part_num, child_part_num, year

"""
).df()

part_qty_per_year


,id,year,parent_part_num,child_part_num,parent_qty,child_qty
0,1112671100,2013,11126,71100,25,0
1,1112671100,2014,11126,71100,13,0
2,1112671100,2022,11126,71100,4,4
3,112085033,2013,11208,5033,12,0
4,112085033,2014,11208,5033,16,0
...,...,...,...,...,...,...
11397,flex08c126644,1999,flex08c12,6644,0,6
11398,flex08c126644,2000,flex08c12,6644,0,6
11399,flex08c126644,2001,flex08c12,6644,0,2
11400,flex08c126644,2003,flex08c12,6644,2,4


In [11]:
part_names_categories= duckdb.sql("""
                             
SELECT p.part_num, p.name, pc.name AS category
FROM parts p JOIN part_categories pc ON p.part_cat_id = pc.id
WHERE p.part_num IN(SELECT parent_part_num FROM part_qty_per_year)
OR p.part_num IN(SELECT child_part_num FROM part_qty_per_year)

""").df()
                             
part_names_categories
                             

,part_num,name,category
0,100559pat0001pr0002,"Animal, Dog, Dachshund with Vibrant Yellow Har...",Animals / Creatures
1,11126,Rip Cord Flexible with Handle,Tools
2,11208,"Wheel 14mm D. x 9.9mm with Centre Groove, Fake...",Wheels and Tyres
3,11213,Plate Round 6 x 6 with Hole,Plates Round Curved and Dishes
4,11299,Ladder 16 x 3.5 with Side Supports,"Bars, Ladders and Fences"
...,...,...,...
460,98459,"Duplo Door / Lid, Wood Effect","Duplo, Quatro and Primo"
461,98562,Large Figure Weapon Claw / Handcuff,Large Buildable Figures
462,98563,"Large Figure Weapon, Zamor Sphere Launcher, To...",Large Buildable Figures
463,flex08c12,Technic Flex Cable 12L,Technic Special


In [71]:
part_classification = duckdb.sql(
""" 
WITH parts AS(
    SELECT parent_part_num AS part_num, year, parent_qty AS qty FROM part_qty_per_year
    UNION
    SELECT child_part_num AS part_num, year, child_qty AS qty  FROM part_qty_per_year
    EXCEPT
    (SELECT parent_part_num, year, parent_qty FROM part_qty_per_year INTERSECT SELECT child_part_num, year, child_qty FROM part_qty_per_year)  
),

running_totals AS(
    SELECT 
        part_num,
        year,
        qty,
        (SUM(qty) OVER (PARTITION BY part_num ORDER BY year))::int AS running_vol,      --this is a cumluative frequency, ehich surmounts to the total_vol on the last row of evary partition
        (SUM(qty) OVER (PARTITION BY part_num))::int AS total_vol,                      --this is a singulaer value repeated across every row, per part
        year - LAG(year) OVER (PARTITION BY part_num ORDER BY year) AS prev_year_gap
    FROM parts
),

cleaned AS( --remove rows that are full of zeros due to the parent-child relationship
    SELECT * FROM running_totals WHERE running_vol / total_vol BETWEEN 0.01 AND 0.99
),

metrics_agg AS(
    SELECT
        part_num,
        total_vol,
        ROUND(COUNT(DISTINCT year) / (MAX(year) - MIN(year) + 1)::DOUBLE, 2) AS consistency     --measures data density over time 

    FROM cleaned
    GROUP BY part_num, total_vol
),

metrics_final AS (
    SELECT 
        part_num,
        ROUND(PERCENT_RANK() OVER (ORDER BY total_vol), 2) AS usage,
        consistency
    FROM metrics_agg
)


SELECT * FROM metrics_final ORDER BY usage DESC, consistency DESC
"""
).df()

part_classification

,part_num,usage,consistency
0,4740,1.00,1.0
1,32123b,1.00,1.0
2,3742,0.99,1.0
3,60474,0.99,1.0
4,60592,0.99,1.0
...,...,...,...
380,88432,0.00,1.0
381,18907,0.00,1.0
382,817c02,0.00,1.0
383,89908,0.00,1.0


In [ ]:
relationship_classification = duckdb.sql(
""" 
WITH metrics AS (
    SELECT 
        id,
        parent_part_num,
        child_part_num,

        --min and max parent part production years
        MIN(CASE WHEN parent_qty > 0 THEN year END) AS parent_min,
        MAX(CASE WHEN parent_qty > 0 THEN year END) AS parent_max,
        
        --min and max child part production years
        MIN(CASE WHEN child_qty > 0 THEN year END) AS child_min,
        MAX(CASE WHEN child_qty > 0 THEN year END) AS child_max,
        
        --will be used to check if part is still in production
        MAX(year) AS max_year


    FROM part_qty_per_year
    GROUP BY id, parent_part_num, child_part_num
),

parent_child_classified AS (
    SELECT 

        *,
        
        CASE
            --co-production needs to be checked first, otherwise the other conditions will take precedence
            WHEN parent_max = max_year AND child_max = max_year
                THEN '3 - child and parent still in production together'

  
            WHEN child_min >= parent_min AND child_max < parent_max
                THEN '4 - child was short-lived within parent lifespan'

   
            WHEN child_min < parent_max AND child_max > parent_max -- Fixed parent_min to parent_max
                THEN '1 - child potentially replaced parent, with lifespan overlap'


            WHEN child_min >= parent_max
                THEN '2 - child potentially replaced parent, without lifespan overlap'

            WHEN child_min < parent_min 
                THEN '5 - child timeline predates parent timeline - review manually'

        END AS category
    FROM metrics
)

SELECT parent_part_num, child_part_num, category  FROM parent_child_classified



""").df()

relationship_classification

,parent_part_num,child_part_num,category
0,132a,3482,"1 - child potentially replaced parent, with li..."
1,14418,22890,3 - child and parent still in production together
2,15118,87913,"1 - child potentially replaced parent, with li..."
3,22380,30090,3 - child and parent still in production together
4,2346,13971,"2 - child potentially replaced parent, without..."
...,...,...,...
480,89201,55981,3 - child and parent still in production together
481,89201,55982,3 - child and parent still in production together
482,92094,31171,5 - child timeline predates parent timeline - ...
483,92402,55981,3 - child and parent still in production together


In [63]:
os.makedirs("report_data", exist_ok=True)
parent_child_parts.to_csv(os.path.join("report_data", "parent_child_parts.csv"), index=False)
part_qty_per_year.to_csv(os.path.join("report_data", "part_qty_per_year.csv"), index=False)
part_names_categories.to_csv(os.path.join("report_data", "part_names_categories.csv"), index=False)

Pieces of information that will help determine a conclusion:

- did the parent part retire before the child part was introduced?
    - if so, what was the period between one part being retired and the ther being introduced?
    - if not, what was the period of time where both parts were in production?
        - if both are retired, did they retire at the same time?
        - are both parts still in production?

- any instances where the parent part was in production longer than the child part?
    - may indicate the 'failure' of a child part...
    - but may be anomalous if the part is hyper-specifc to one set or scenario, or if the part is relatively new.


- in years where a parent part and a child part coexisted, which was used more in sets? (using the part_qty_per_year table)
    - bar-chart timelines, including drill-downs by category, of parent part quantity vs child part

- 're-introduction' of a part that may have been perceived as discontinued. Set an arbitrary time gap, eg. 5 years.
